In [1]:

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


# client = OpenAI()


/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [6]:
import pandas as pd

summary = []

for col in df.columns:
    summary.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "num_unique": df[col].nunique(dropna=True),
        "num_missing": df[col].isna().sum()
    })

summary_df = pd.DataFrame(summary)

# Display table
remove_cols = ["Year", "Month", "Day", "Time", "Amount","Is Fraud?","User"]

# Filter them out
summary_df = summary_df[~summary_df["column"].isin(remove_cols)]
summary_df = summary_df.reset_index(drop=True)


# Display final table
print(summary_df)

           column    dtype  num_unique  num_missing
0            Card    int64           9            0
1        Use Chip   object           3            0
2   Merchant Name    int64      100343            0
3   Merchant City   object       13429            0
4  Merchant State   object         223      2720821
5             Zip  float64       27321      2878135
6             MCC    int64         109            0
7         Errors?   object          23     23998469


In [11]:
# Fraud = 1, Legit = 0

df_fraud = df[df["Is Fraud?"] == "Yes"].copy()
df_legit = df[df["Is Fraud?"] == "No"].copy()

print("Fraud shape:", df_fraud.shape)
print("Legit shape:", df_legit.shape)

Fraud shape: (29757, 15)
Legit shape: (24357143, 15)


In [13]:
MIN_UNIQUE_VALUES = 20
TOP_N = 20

print("\n" + "=" * 60)
print("FREQUENCY COMPARISON (FRAUD vs LEGIT)")
print("=" * 60)

frequency_tables = {}

for col in df_fraud.columns:

    # Replace NaN with label
    fraud_series = df_fraud[col].where(df_fraud[col].notna(), "<MISSING>")
    legit_series = df_legit[col].where(df_legit[col].notna(), "<MISSING>")

    fraud_freq = fraud_series.value_counts(dropna=False)
    legit_freq = legit_series.value_counts(dropna=False)

    combined = pd.concat(
        [
            fraud_freq.rename("Fraud_count"),
            legit_freq.rename("Legit_count")
        ],
        axis=1
    ).fillna(0)

    total_fraud = len(df_fraud)
    total_legit = len(df_legit)

    combined["Fraud_%"] = (combined["Fraud_count"] / total_fraud * 100).round(4)
    combined["Legit_%"] = (combined["Legit_count"] / total_legit * 100).round(4)

    # Add difference column (VERY IMPORTANT for insights)
    combined["Diff_%"] = (combined["Fraud_%"] - combined["Legit_%"]).round(4)

    # Sort by strongest fraud signal
    combined = combined.sort_values(by="Diff_%", ascending=False)

    # Skip high-cardinality columns (too noisy)
    # if len(combined) > 1000:
    #     print(f"\nSkipping Column: {col} (High Cardinality: {len(combined)})")
    #     continue

    frequency_tables[col] = combined

    print(f"\nColumn: {col} | Unique values: {len(combined)}")
    display(combined.head(TOP_N))

    if len(combined) > TOP_N:
        print(f"Showing top {TOP_N} rows out of {len(combined)}. Full table in frequency_tables['{col}']")


FREQUENCY COMPARISON (FRAUD vs LEGIT)

Column: User | Unique values: 2000


,Fraud_count,Legit_count,Fraud_%,Legit_%,Diff_%
User,,,,,
1487,110.0,18813,0.3697,0.0772,0.2925
1709,86.0,10132,0.2890,0.0416,0.2474
1425,100.0,22762,0.3361,0.0935,0.2426
190,82.0,12432,0.2756,0.0510,0.2246
945,75.0,7459,0.2520,0.0306,0.2214
615,81.0,15070,0.2722,0.0619,0.2103
1002,74.0,10797,0.2487,0.0443,0.2044
914,83.0,18367,0.2789,0.0754,0.2035
1122,76.0,13116,0.2554,0.0538,0.2016


Showing top 20 rows out of 2000. Full table in frequency_tables['User']

Column: Card | Unique values: 9


,Fraud_count,Legit_count,Fraud_%,Legit_%,Diff_%
Card,,,,,
3,4135,2786650,13.8959,11.4408,2.4551
4,2157,1306963,7.2487,5.3658,1.8829
2,5807,4299787,19.5147,17.6531,1.8616
5,1016,562081,3.4143,2.3077,1.1066
6,350,176379,1.1762,0.7241,0.4521
7,109,46274,0.3663,0.1900,0.1763
8,25,5159,0.0840,0.0212,0.0628
1,7514,6486083,25.2512,26.6291,-1.3779
0,8644,8687767,29.0486,35.6683,-6.6197



Column: Year | Unique values: 30


...

,Fraud_count,Legit_count,Fraud_%,Legit_%,Diff_%
MCC,,,,,
5311,4824.0,880896,16.2113,3.6166,12.5947
5310,2152.0,451955,7.2319,1.8555,5.3764
5300,2201.0,1120836,7.3966,4.6017,2.7949
5732,843.0,11750,2.8329,0.0482,2.7847
5815,879.0,114368,2.9539,0.4695,2.4844
5651,849.0,136640,2.8531,0.5610,2.2921
5094,567.0,8877,1.9054,0.0364,1.8690
5719,717.0,159560,2.4095,0.6551,1.7544
5045,470.0,4543,1.5795,0.0187,1.5608


Showing top 20 rows out of 109. Full table in frequency_tables['MCC']

Column: Errors? | Unique values: 24


,Fraud_count,Legit_count,Fraud_%,Legit_%,Diff_%
Errors?,,,,,
"Bad CVV,",280.0,10460,0.9410,0.0429,0.8981
"Bad PIN,",302.0,58616,1.0149,0.2407,0.7742
"Bad Expiration,",120.0,10596,0.4033,0.0435,0.3598
"Insufficient Balance,",396.0,242387,1.3308,0.9951,0.3357
"Bad Card Number,",105.0,13216,0.3529,0.0543,0.2986
"Bad PIN,Insufficient Balance,",7.0,574,0.0235,0.0024,0.0211
"Technical Glitch,",63.0,48094,0.2117,0.1975,0.0142
"Bad CVV,Insufficient Balance,",4.0,85,0.0134,0.0003,0.0131
"Bad Expiration,Technical Glitch,",2.0,30,0.0067,0.0001,0.0066


Showing top 20 rows out of 24. Full table in frequency_tables['Errors?']

Column: Is Fraud? | Unique values: 2


,Fraud_count,Legit_count,Fraud_%,Legit_%,Diff_%
Is Fraud?,,,,,
Yes,29757.0,0.0,100.0,0.0,100.0
No,0.0,24357143.0,0.0,100.0,-100.0


In [7]:
print(df.columns)

print("\nUnique values in target column:")
print(df["Is Fraud?"].unique())

print("\nValue counts:")
print(df["Is Fraud?"].value_counts(dropna=False))

Index(['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip',
       'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC',
       'Errors?', 'Is Fraud?'],
      dtype='object')

Unique values in target column:
['No' 'Yes']

Value counts:
Is Fraud?
No     24357143
Yes       29757
Name: count, dtype: int64


In [13]:
import pandas as pd

MIN_UNIQUE_VALUES = 20
TOP_N = 20

print("\n" + "=" * 60)
print("STEP 1: FIX TARGET COLUMN")
print("=" * 60)

# Convert Yes/No → 1/0
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

print(df["Is Fraud?"].value_counts())

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 2: SPLIT DATA")
print("=" * 60)

fraud_df = df[df["Is Fraud?"] == 1]
nonfraud_df = df[df["Is Fraud?"] == 0]

print(f"Fraud shape: {fraud_df.shape}")
print(f"Non-Fraud shape: {nonfraud_df.shape}")

# --------------------------------------------------

def generate_frequency_tables(data, label):
    print("\n" + "=" * 60)
    print(f"FREQUENCY TABLES FOR: {label}")
    print("=" * 60)

    frequency_tables = {}

    for col in data.columns:
        series = data[col].astype("object").where(data[col].notna(), "<MISSING>")

        freq_table = (
            series.value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="count")
        )

        freq_table["percentage"] = (
            freq_table["count"] / len(data) * 100
        ).round(4)

        frequency_tables[col] = freq_table

        print(f"\nColumn: {col} | Unique values: {len(freq_table)}")
        display(freq_table.head(TOP_N))

        if len(freq_table) > TOP_N:
            print(f"Showing top {TOP_N} rows out of {len(freq_table)}")

    return frequency_tables

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3: GENERATE FREQUENCY TABLES")
print("=" * 60)

fraud_tables = generate_frequency_tables(fraud_df, "FRAUD DATA")
nonfraud_tables = generate_frequency_tables(nonfraud_df, "NON-FRAUD DATA")

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 4: FRAUD vs NON-FRAUD COMPARISON (IMPORTANT)")
print("=" * 60)

def compare_feature(col):
    f = fraud_df[col].value_counts(normalize=True)
    nf = nonfraud_df[col].value_counts(normalize=True)

    comparison = pd.concat(
        [f, nf], axis=1, keys=["fraud_pct", "nonfraud_pct"]
    ).fillna(0)

    comparison["diff"] = comparison["fraud_pct"] - comparison["nonfraud_pct"]

    return comparison.sort_values("diff", ascending=False)

# Example comparisons (change columns as needed)
important_cols = [
    "Merchant Name",
    "Merchant City",
    "MCC",
    "Use Chip",
    "Errors?"
]

comparison_results = {}

for col in df.columns:
    print(f"\n🔍 Comparing: {col}")
    comp = compare_feature(col)
    display(comp.head(20))
    comparison_results[col] = comp

# --------------------------------------------------

print("\n" + "=" * 60)
print("DONE ✅")
print("=" * 60)


STEP 1: FIX TARGET COLUMN
Series([], Name: count, dtype: int64)

STEP 2: SPLIT DATA
Fraud shape: (0, 15)
Non-Fraud shape: (0, 15)

STEP 3: GENERATE FREQUENCY TABLES

FREQUENCY TABLES FOR: FRAUD DATA

Column: User | Unique values: 0


,User,count,percentage



Column: Card | Unique values: 0


,Card,count,percentage



Column: Year | Unique values: 0


...


🔍 Comparing: Errors?


,fraud_pct,nonfraud_pct,diff
Errors?,,,



🔍 Comparing: Is Fraud?


,fraud_pct,nonfraud_pct,diff
Is Fraud?,,,



DONE ✅


In [16]:
import pandas as pd

MIN_UNIQUE_VALUES = 2
MAX_UNIQUE_VALUES = 50
TOP_N = 20

print("\n" + "=" * 60)
print("STEP 1: FIX TARGET COLUMN")
print("=" * 60)

# Convert Yes/No → 1/0
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

print(df["Is Fraud?"].value_counts())

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 2: CLEAN DATA (IMPORTANT)")
print("=" * 60)

# Clean all object columns
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()      # remove spaces
            .str.lower()      # normalize case
        )

# Fill missing values globally
df = df.fillna("<MISSING>")

print("✅ Data cleaned")

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3: SPLIT DATA")
print("=" * 60)

fraud_df = df[df["Is Fraud?"] == 1]
nonfraud_df = df[df["Is Fraud?"] == 0]

print(f"Fraud shape: {fraud_df.shape}")
print(f"Non-Fraud shape: {nonfraud_df.shape}")

# --------------------------------------------------

def generate_frequency_tables(data, label):
    print("\n" + "=" * 60)
    print(f"FREQUENCY TABLES FOR: {label}")
    print("=" * 60)

    frequency_tables = {}

    for col in data.columns:
        series = data[col]

        freq_table = (
            series.value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="count")
        )

        freq_table["percentage"] = (
            freq_table["count"] / len(data) * 100
        ).round(4)

        frequency_tables[col] = freq_table

        print(f"\nColumn: {col} | Unique values: {len(freq_table)}")
        display(freq_table.head(TOP_N))

        if len(freq_table) > TOP_N:
            print(f"Showing top {TOP_N} rows out of {len(freq_table)}")

    return frequency_tables

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 4: GENERATE FREQUENCY TABLES")
print("=" * 60)

fraud_tables = generate_frequency_tables(fraud_df, "FRAUD DATA")
nonfraud_tables = generate_frequency_tables(nonfraud_df, "NON-FRAUD DATA")

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 5: FRAUD vs NON-FRAUD COMPARISON")
print("=" * 60)

def compare_feature(col):
    f = fraud_df[col].value_counts(normalize=True)
    nf = nonfraud_df[col].value_counts(normalize=True)

    comparison = pd.concat(
        [f, nf], axis=1, keys=["fraud_pct", "nonfraud_pct"]
    ).fillna(0)

    comparison["diff"] = comparison["fraud_pct"] - comparison["nonfraud_pct"]
    comparison["abs_diff"] = comparison["diff"].abs()

    return comparison.sort_values("abs_diff", ascending=False)


comparison_results = {}

for col in df.columns:

    # Skip target column
    if col == "Is Fraud?":
        continue

    # Skip numeric columns
    if pd.api.types.is_numeric_dtype(df[col]):
        continue

    unique_vals = df[col].nunique()

    # Skip low/high useless columns
    if unique_vals < MIN_UNIQUE_VALUES or unique_vals > MAX_UNIQUE_VALUES:
        continue

    print(f"\n🔍 Comparing: {col} (Unique: {unique_vals})")

    comp = compare_feature(col)

    display(comp.head(TOP_N))

    comparison_results[col] = comp

print("\n" + "=" * 60)
print("DONE ✅ (All meaningful columns compared)")
print("=" * 60)


STEP 1: FIX TARGET COLUMN
Series([], Name: count, dtype: int64)

STEP 2: CLEAN DATA (IMPORTANT)


KeyboardInterrupt: 

In [10]:
import pandas as pd

TOP_N = 10
DIFF_THRESHOLD = 0.02  # 2% difference

print("\n" + "=" * 60)
print("STEP 1: CLEAN + FIX TARGET COLUMN")
print("=" * 60)

# Clean + normalize values
df["Is Fraud?"] = (
    df["Is Fraud?"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Map to numeric
df["Is Fraud?"] = df["Is Fraud?"].map({
    "yes": 1,
    "no": 0
})

# Safety check
print("\nValue counts after mapping:")
print(df["Is Fraud?"].value_counts(dropna=False))

# Ensure no NaNs
if df["Is Fraud?"].isna().sum() > 0:
    raise ValueError("❌ Mapping failed: NaN values present in 'Is Fraud?'")

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 2: SPLIT DATA")
print("=" * 60)

fraud_df = df[df["Is Fraud?"] == 1]
nonfraud_df = df[df["Is Fraud?"] == 0]

print(f"Fraud shape: {fraud_df.shape}")
print(f"Non-Fraud shape: {nonfraud_df.shape}")

# --------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3: ANOMALY TABLE (COMBINED VIEW)")
print("=" * 60)

final_results = {}

for col in df.columns:
    if col == "Is Fraud?":
        continue

    # Compute normalized distributions
    f = fraud_df[col].value_counts(normalize=True)
    nf = nonfraud_df[col].value_counts(normalize=True)

    comp = pd.concat(
        [f, nf], axis=1, keys=["fraud_pct", "nonfraud_pct"]
    ).fillna(0)

    # Difference
    comp["diff"] = comp["fraud_pct"] - comp["nonfraud_pct"]

    # Filter anomalies
    comp = comp[comp["diff"].abs() > DIFF_THRESHOLD]

    if len(comp) > 0:
        comp = comp.sort_values("diff", ascending=False).head(TOP_N)
        comp = comp.reset_index().rename(columns={"index": col})

        final_results[col] = comp

        print(f"\n🚨 Column: {col}")
        display(comp)

# --------------------------------------------------

print("\n" + "=" * 60)
print(f"TOTAL COLUMNS WITH ANOMALIES: {len(final_results)}")
print("=" * 60)


STEP 1: CLEAN + FIX TARGET COLUMN

Value counts after mapping:
Is Fraud?
NaN    24386900
Name: count, dtype: int64


ValueError: ❌ Mapping failed: NaN values present in 'Is Fraud?'

In [3]:

df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

df["Errors?"] = (df["Errors?"] != "None").astype(int)

df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

df = df.fillna(0)

# print(df.dtypes)
# print(df.head())

/tmp/ipykernel_742035/3282332532.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [4]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [5]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

## Run form here

In [6]:

from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/tmp/ipykernel_742035/2009901296.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
for m in genai.list_models():
    print(m.name, "->", m.supported_generation_methods)

models/gemini-2.5-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts -> ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts -> ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it -> ['generateContent', 'countTokens']
models/gemma-3-4b-it -> ['generateContent', 'countTokens']
models/gemma-3-12b-it -> ['generateContent

In [ ]:
model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")


In [9]:
PROMPT = """
==============================
DATA SUMMARY
==============================

- Number of users: 2000
- Transactions span: multiple years (1991–2020)
- Amount has high variability (~100K unique values)
- Merchant Name is high-cardinality (~100K unique)
- Merchant City is high-cardinality (~13K)
- Merchant State includes global locations (223 unique)
- Errors column contains multiple error types (including combinations)
- Fraud label is binary (Yes/No)

==============================
VALUE DISTRIBUTION INSIGHTS
==============================

Use Chip:
- Swipe Transaction
- Online Transaction
- Chip Transaction

Errors?:
- NaN (no error)
- Technical Glitch
- Insufficient Balance
- Bad PIN
- Bad CVV
- Bad Zipcode
- Combinations of above errors

Merchant State:
- Mix of US states and international locations

Is Fraud:
- Yes / No

==============================
SAMPLE DATA (STRUCTURE)
==============================

Each row represents a transaction:

User=0, Card=0, Year=2002, Month=9, Day=1, Time=06:21,
Amount=$134.09, Use Chip=Swipe Transaction,
Merchant Name=high-cardinality ID,
Merchant City=La Verne, Merchant State=CA,
Zip=91750, MCC=5300, Errors?=NaN, Is Fraud=No

==============================
PREPROCESSING & MISSING VALUE HANDLING
==============================

Before feature computation, assume the following preprocessing has been applied:

1. Amount:
   - Remove "$" and convert to float
   - Missing values (if any) → fill with median Amount

2. Time:
   - Convert "HH:MM" → Hour (0–23)
   - Missing values → fill with mode Hour

3. Use Chip:
   - Encode as:
       Swipe Transaction → 0
       Chip Transaction → 1
       Online Transaction → 2
   - Missing values → fill with most frequent category

4. Errors?:
   - NaN → 0 (no error)
   - Any error string → 1 (error present)

5. Merchant City / Merchant State:
   - Missing values → fill with "Unknown"

6. Zip:
   - Missing values → fill with median Zip

7. MCC:
   - Missing values → fill with mode MCC

8. Is Fraud:
   - Yes → 1, No → 0
   - MUST NOT be used for feature generation

After preprocessing, the following cleaned columns are available:

- Amount (float)
- Hour (int)
- Use Chip (int)
- Errors? (int)
- MCC (int)
- Merchant City (string)
- Merchant State (string)
- Zip (float)


==============================
IMPORTANT DATA CHARACTERISTICS
==============================

- Merchant Name is extremely high-cardinality → avoid direct usage
- Merchant City/State have high diversity → use aggregation (nunique)
- Time is granular → useful for behavioral patterns
- Errors column contains multiple combined error types
- Amount has high variance → useful for anomaly detection



============================== 
YOUR TASK 
============================== 
Design USER-LEVEL features for fraud detection. Each feature MUST produce ONE value per User.

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

Return a JSON list:

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "intuition": "...",
    "type": "numeric"
  }
]

- pandas_code must be executable
- must return a Series indexed by User
- no explanations outside JSON

==============================
EXECUTION REQUIREMENTS
==============================

- Features must be computed using pandas
- Final output must be a Series indexed by User
- Length must equal number of unique users
- Must not use undefined variables
- Must not use target column (Is Fraud)

==============================
ALLOWED OPERATIONS
==============================

You may use:

- multiple aggregations
- ratios (e.g., sum / count)
- differences (max - mean)
- standard deviation and variance
- conditional aggregations
- arithmetic combinations of features

==============================
QUALITY CONSTRAINTS
==============================

- Avoid redundant or highly correlated features
- Prefer interpretable features
- Prefer features that capture behavioral differences
- Avoid trivial features (e.g., constant or near-constant)

==============================
FINAL INSTRUCTIONS
==============================

- Generate 12–15 features
- Each feature must be unique and meaningful
- Output ONLY JSON
- Do NOT include explanations outside JSON

Ensure final output remains a valid Series per User.
"""

In [10]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [11]:
# response = call_llm(PROMPT)

response = model_llm.generate_content(PROMPT)
print("RAW OUTPUT:\n", response)

RAW OUTPUT:
 response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "```json\n[\n  {\n    \"feature_name\": \"user_avg_transaction_amount\",\n    \"pandas_code\": \"df.groupby('User')['Amount'].mean()\",\n    \"description\": \"Average transaction amount for the user across all time.\",\n    \"intuition\": \"Legitimate users typically have a consistent average spending level. Outliers might indicate anomalous spending behavior.\",\n    \"type\": \"numeric\"\n  },\n  {\n    \"feature_name\": \"user_std_transaction_amount\",\n    \"pandas_code\": \"df.groupby('User')['Amount'].std().fillna(0)\",\n    \"description\": \"Standard deviation of transaction amounts for the user.\",\n    \"intuition\": \"High standard deviation suggests highly variable spending habits (large vs. small purchases), which might be character

In [12]:
import re

def fix_code(code):
    # Fix missing quotes
    replacements = {
        "[Amount]": "['Amount']",
        "[Time]": "['Time']",
        "[Is Fraud?]": "['Is Fraud?']"
    }
    for k, v in replacements.items():
        code = code.replace(k, v)

    # 🔥 Fix tuple column selection → list
    code = re.sub(
        r"\[['\"](\w+)['\"],\s*['\"](\w+)['\"]\]",
        r"[[ '\1', '\2' ]]",
        code
    )

    # Also fix ('A','B') → ['A','B']
    code = re.sub(
        r"\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)",
        r"['\1', '\2']",
        code
    )

    return code

In [13]:
def extract_json(llm_output):
    try:
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []
    


In [14]:
import re

ALLOWED_AGGS = ["mean", "sum", "count", "nunique", "max", "min", "std"]

def fix_code(code):
    if not code:
        return None

    code = code.strip()

    # =========================
    # 1. Fix missing quotes for columns
    # =========================
    code = re.sub(r"\[(\w[\w\s?]*)\]", r"['\1']", code)

    # =========================
    # 2. Fix tuple → list
    # =========================
    code = re.sub(r"\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)", r"['\1', '\2']", code)

    # =========================
    # 3. Normalize spaces
    # =========================
    code = re.sub(r"\s+", " ", code)

    # =========================
    # 4. MUST contain correct groupby
    # =========================
    if "df.groupby('User')" not in code:
        return None

    # =========================
    # 5. Reject multi-column groupby
    # =========================
    if "groupby(['User'" in code or 'groupby(["User"' in code:
        return None

    # =========================
    # 6. Reject multiple groupby
    # =========================
    if code.count("groupby") > 1:
        return None

    # =========================
    # 7. Reject forbidden ops
    # =========================
    forbidden = [
        "reset_index", "apply", "map", "values",
        "resample", "[[", "]]"
    ]
    if any(f in code for f in forbidden):
        return None

    # =========================
    # 8. Reject arithmetic (multiple aggregations)
    # =========================
    if any(op in code for op in ["/", "+", "-", "*"]):
        return None

    # =========================
    # 9. Validate aggregation
    # =========================
    agg_match = re.search(r"\.(\w+)\(\)", code)
    if not agg_match:
        return None

    agg = agg_match.group(1)
    if agg not in ALLOWED_AGGS and "size()" not in code:
        return None

    # =========================
    # 10. Final strict pattern check
    # =========================
    pattern1 = r"^df\.groupby\('User'\)\['[^']+'\]\.(mean|sum|count|nunique|max|min|std)\(\)$"
    pattern2 = r"^df\.groupby\('User'\)\.size\(\)$"

    if not (re.match(pattern1, code) or re.match(pattern2, code)):
        return None

    return code

In [15]:
# # response = model_llm.generate_content(PROMPT)
# features = extract_json(response)


# user_df = pd.DataFrame()

# user_df = pd.DataFrame(index=df['User'].unique())

# for f in features:
#     try:
#         code = fix_code(f.get("pandas_code", ""))

#         if not code:
#             print("Skipping invalid:", f["feature_name"])
#             continue

#         print("Applying:", f["feature_name"])

#         result = eval(code)

#         if not isinstance(result, pd.Series):
#             raise ValueError("Not a pandas Series")

#         result = result.reindex(user_df.index)

#         user_df[f["feature_name"]] = result
        
#     except Exception as e:
#         print("Error in", f["feature_name"], ":", e)
        

# user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()
# user_df["is_fraud_user"] = user_df["is_fraud_user"].reindex(user_df.index)

# user_df = user_df.fillna(0)

# # for f in features:
# #     try:
# #         print("Applying:", f["feature_name"])
# #         user_df[f["feature_name"]] = eval(f["pandas_code"])
# #     except Exception as e:
# #         print("Error in", f["feature_name"], ":", e)

# # user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# # user_df = user_df.fillna(0)



In [16]:
response = model_llm.generate_content(PROMPT)

output = response.text
# print("Raw Output:\n", output)

output = output.replace("```json", "").replace("```", "").strip()

features = json.loads(output)

# =========================================
# ✅ APPLY FEATURES USING PANDAS
# =========================================

# Example: load your dataset
# df = pd.read_csv("your_dataset.csv")

user_df = pd.DataFrame()

for f in features:
    try:
        print("Applying:", f["feature_name"])
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)

# ✅ Add label
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# ✅ Fill missing
user_df = user_df.fillna(0)

print("\nFinal Shape:", user_df.shape)
print(user_df.head())

Applying: user_avg_transaction_amount
Applying: user_std_transaction_amount
Applying: user_transaction_count
Applying: user_online_ratio
Applying: user_chip_transaction_ratio
Applying: user_error_rate
Applying: user_time_of_day_variance
Applying: user_unique_merchant_city_count
Applying: user_unique_merchant_state_count
Applying: user_median_zip_change_rate
Applying: user_avg_time_since_first_transaction
Applying: user_time_weighted_amount_sum
Error in user_time_weighted_amount_sum : invalid syntax (<string>, line 1)
Applying: user_mcc_diversity
Applying: user_avg_amount_vs_std_ratio

Final Shape: (2000, 14)
      user_avg_transaction_amount  user_std_transaction_amount  \
User                                                             
0                       81.299989                    94.159093   
1                       81.118050                   156.784691   
2                       35.159687                    54.298552   
3                      117.277603                   26

In [17]:
print("\nFinal Shape:", user_df.shape)
print(user_df.head())


Final Shape: (2000, 14)
      user_avg_transaction_amount  user_std_transaction_amount  \
User                                                             
0                       81.299989                    94.159093   
1                       81.118050                   156.784691   
2                       35.159687                    54.298552   
3                      117.277603                   268.654472   
4                       97.011698                   138.619564   

      user_transaction_count  user_online_ratio  user_chip_transaction_ratio  \
User                                                                           
0                      19963                0.0                          0.0   
1                       8919                0.0                          0.0   
2                      41978                0.0                          0.0   
3                      10117                0.0                          0.0   
4                      18542    

In [18]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [19]:
print(user_df.shape)

(2000, 14)


In [20]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [21]:
print(y.isna().sum())

0


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [23]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.89      0.73      0.80       131
           1       0.88      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.84      0.86       400
weighted avg       0.88      0.88      0.88       400

ROC-AUC: 0.9071767076250745


In [25]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)

user_transaction_count                   0.292742
user_unique_merchant_city_count          0.165376
user_unique_merchant_state_count         0.159582
user_mcc_diversity                       0.140691
user_avg_time_since_first_transaction    0.067499
user_std_transaction_amount              0.049799
user_avg_amount_vs_std_ratio             0.044967
user_avg_transaction_amount              0.041288
user_median_zip_change_rate              0.038056
user_chip_transaction_ratio              0.000000
user_online_ratio                        0.000000
user_error_rate                          0.000000
user_time_of_day_variance                0.000000
dtype: float64


In [26]:
response = model_llm.generate_content(PROMPT)

output = response.text
# print("Raw Output:\n", output)

output = output.replace("```json", "").replace("```", "").strip()

features = json.loads(output)

# =========================================
# ✅ APPLY FEATURES USING PANDAS
# =========================================

# Example: load your dataset
# df = pd.read_csv("your_dataset.csv")

user_df = pd.DataFrame()

for f in features:
    try:
        print("Applying:", f["feature_name"])
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)

# ✅ Add label
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# ✅ Fill missing
user_df = user_df.fillna(0)

print("\nFinal Shape:", user_df.shape)
print(user_df.head())

Applying: user_avg_transaction_amount
Applying: user_std_transaction_amount
Applying: user_transaction_count
Applying: user_velocity_7_day
Error in user_velocity_7_day : invalid syntax (<string>, line 1)
Applying: user_chip_usage_ratio
Error in user_chip_usage_ratio : invalid syntax (<string>, line 1)
Applying: user_error_rate
Error in user_error_rate : invalid syntax (<string>, line 1)
Applying: user_time_of_day_skew
Error in user_time_of_day_skew : invalid syntax (<string>, line 1)
Applying: user_avg_amount_vs_all_median
Error in user_avg_amount_vs_all_median : invalid syntax (<string>, line 1)
Applying: user_merchant_city_diversity
Error in user_merchant_city_diversity : invalid syntax (<string>, line 1)
Applying: user_merchant_state_diversity
Error in user_merchant_state_diversity : invalid syntax (<string>, line 1)
Applying: user_ratio_unusual_state
Error in user_ratio_unusual_state : invalid syntax (<string>, line 1)
Applying: user_zip_code_entropy
Error in user_zip_code_entropy 

In [27]:
feature_names = [f["feature_name"] for f in features]

In [28]:
top_features = importance.to_dict()

In [29]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

results= {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}
with open("results_2.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved to results_2.json")

Saved to results_2.json


In [30]:
print(report1)
print(importance)
print(roc1)

              precision    recall  f1-score   support

           0       0.89      0.73      0.80       131
           1       0.88      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.84      0.86       400
weighted avg       0.88      0.88      0.88       400

user_transaction_count                   0.292742
user_unique_merchant_city_count          0.165376
user_unique_merchant_state_count         0.159582
user_mcc_diversity                       0.140691
user_avg_time_since_first_transaction    0.067499
user_std_transaction_amount              0.049799
user_avg_amount_vs_std_ratio             0.044967
user_avg_transaction_amount              0.041288
user_median_zip_change_rate              0.038056
user_chip_transaction_ratio              0.000000
user_online_ratio                        0.000000
user_error_rate                          0.000000
user_time_of_day_variance                0.000000
dtype: float64
0.907176

In [31]:
sample = df.sample(n=5, random_state=42)
print(sample)
sample_str = sample.to_string(index=False)

          User  Card  Year  Month  Day  Time  Amount  Use Chip  \
18199893  1470     0  2019      7   10   0.0   59.18       0.0   
9731325    822     1  2019      1   14   0.0  280.91       0.0   
536687      41     3  2010      3   15   0.0 -144.00       0.0   
13223840  1084     0  2015      9   20   0.0    6.76       0.0   
17070521  1384     0  2014     10   12   0.0    9.17       0.0   

                Merchant Name Merchant City Merchant State      Zip   MCC  \
18199893 -6853385250336487907       Harwood             MD  20776.0  5813   
9731325   4241336128694185533        ONLINE              0      0.0  4814   
536687     190253443608377572         Hemet             CA  92543.0  3359   
13223840 -7837310524365334241     Littleton             CO  80122.0  5300   
17070521 -5023497618971072366       Gardner             KS  66030.0  5812   

          Errors?  Is Fraud?  Hour  
18199893        1          0   0.0  
9731325         1          0   0.0  
536687          1          0 

In [32]:
def get_prompt(llm_input, df):
    sample = df.sample(n=5, random_state=42)
    sample_str = sample.to_string(index=False)

    PROMPT = f"""
You are a fraud analytics feature engineering expert.

A RandomForestClassifier was trained on USER-LEVEL features.

Your goal is to IMPROVE feature quality based on previous model performance.

==============================
DATA SAMPLE (RANDOM)
==============================
{sample_str}

==============================
MODEL PERFORMANCE
==============================

Classification Report:
{llm_input["classification_report"]}

ROC-AUC:
{llm_input["roc_auc"]}

Top Important Features:
{llm_input["top_features"]}

==============================
DATA SUMMARY
==============================

- Number of users: 2000
- Transactions span: multiple years (1991–2020)
- Amount has high variability (~100K unique values)
- Merchant Name is high-cardinality (~100K unique) → DO NOT USE
- Merchant City is high-cardinality (~13K)
- Merchant State includes global locations (223 unique)
- Errors column contains multiple error types (including combinations)
- Fraud label is binary (Yes/No)

==============================
VALUE DISTRIBUTION INSIGHTS
==============================

Use Chip:
- Swipe Transaction
- Online Transaction
- Chip Transaction

Errors?:
- NaN (no error)
- Technical Glitch
- Insufficient Balance
- Bad PIN
- Bad CVV
- Bad Zipcode
- Combinations of above errors

Merchant State:
- Mix of US states and international locations

Is Fraud:
- Yes / No

==============================
PREPROCESSING ASSUMPTIONS
==============================

Assume the following preprocessing has already been applied:

- Amount → float
- Time → Hour (0–23)
- Use Chip → encoded (0/1/2)
- Errors? → binary (0 = no error, 1 = error)
- Is Fraud → binary (DO NOT USE)

Available columns:

User, Card, Year, Month, Day, Hour, Amount,
Use Chip, Merchant City, Merchant State,
Zip, MCC, Errors?

==============================
CRITICAL GOAL
==============================

Generate USER-LEVEL behavioral features.

Each feature MUST produce:
👉 EXACTLY ONE VALUE PER USER

==============================
YOUR TASK
==============================

1. Analyze model performance:
   - Identify weaknesses (especially fraud recall)
   - Identify missing behavioral signals

2. Improve feature space:
   - Generate NEW features
   - Avoid duplicating existing top features

3. Focus on features that increase discrimination between fraud and non-fraud users

==============================
FEATURE DESIGN FOCUS
==============================

- abnormal spending patterns
- spending variability (std, max, min)
- transaction frequency / bursts
- merchant/category diversity
- geographic spread
- temporal irregularity (Hour patterns)
- error behavior
- chip vs swipe usage
- anomaly detection signals

==============================
ALLOWED OPERATIONS
==============================

You MAY use:

- multiple aggregations
- ratios (e.g., sum / count)
- differences (max - mean)
- standard deviation
- conditional aggregations
- arithmetic combinations

==============================
STRICT CODE RULES
==============================

- All code must use dataframe 'df'
- All computations must be based on df.groupby('User')
- Output must be a pandas Series indexed by User
- Do NOT use .values or numpy conversion
- Do NOT reset index
- Do NOT use apply()
- Do NOT use Merchant Name
- Do NOT use target column (Is Fraud)
- Ensure safe operations (avoid division by zero)
- Write code as a single expression (no intermediate variables)

==============================
QUALITY CONSTRAINTS
==============================

- Avoid redundant features
- Avoid highly correlated features
- Prefer interpretable features
- Prefer features capturing behavioral differences
- Avoid trivial features
- Replace or improve weak/low-importance patterns


==============================
OUTPUT FORMAT (STRICT JSON)
==============================

[
  {{
    "feature_name": "...",
    "pandas_code": "...",
    "reason": "Why this feature improves fraud detection"
  }}
]

==============================
FINAL INSTRUCTIONS
==============================

- Generate 15–25 high-quality features
- Each feature must be unique and meaningful
- Every pandas_code must be executable
- Output ONLY JSON
- Do NOT include explanations outside JSON

"""
    return PROMPT

In [45]:
model_llm = genai.GenerativeModel("models/gemini-flash-latest")


In [46]:
def clean_features(X):
    # Replace inf → NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # Fill NaN with 0 (or better: median)
    X = X.fillna(0)

    # Optional: clip extreme values (very important)
    X = X.clip(-1e10, 1e10)

    return X

In [47]:
def split_data(user_df):
    X = user_df.drop(columns=["is_fraud_user"])
    y = user_df["is_fraud_user"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    return X, y, X_train, X_test, y_train, y_test
def train_model(X_train, y_train):
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    return model

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred))

    print("\n=== ROC AUC ===")
    print(roc_auc_score(y_test, y_prob))

    return y_pred, y_prob
def get_feature_importance(model, X):
    
    importance = pd.Series(model.feature_importances_, index=X.columns)
    importance = importance.sort_values(ascending=False)

    print("\n=== Top Features ===")
    print(importance.head(10))

    return importance

In [48]:
def get_userdf(llm_input):
    PROMPT = get_prompt(llm_input,df)

    response = model_llm.generate_content(PROMPT)

    output = response.text
    # print("Raw Output:\n", output)

    output = output.replace("```json", "").replace("```", "").strip()

    features = json.loads(output)


    user_df = pd.DataFrame()

    for f in features:
        try:
            print("Applying:", f["feature_name"])
            user_df[f["feature_name"]] = eval(f["pandas_code"])
        except Exception as e:
            print("Error in", f["feature_name"], ":", e)

    # ✅ Add label
    user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

    # ✅ Fill missing
    user_df = user_df.fillna(0)
    return user_df
    

In [49]:


def run(user_df):
    
    X, y, X_train, X_test, y_train, y_test = split_data(user_df)
    X = clean_features(X)

    model = train_model(X_train, y_train)

    print("\n=== Model Evaluation ===")
    y_pred, y_prob = evaluate_model(model, X_test, y_test)
    
    report1 = classification_report(y_test, y_pred)
    roc1 = roc_auc_score(y_test, y_prob)
    importance = get_feature_importance(model, X)

    # print("\n=== Top Features ===")
    # print(importance)
    top_features = importance.to_dict()
    feature_names = [f["feature_name"] for f in features]
    
    llm_input = {
        "existing_features": feature_names,
        "classification_report": report1,
        "roc_auc": roc1,
        "top_features": top_features
    }
    return llm_input


In [50]:
# import json

# # # 📂 Load results file (handles multiple JSON blocks)
# # results_list = []

# # with open("results.json", "r") as f:
# #     content = f.read().strip()
# #     # Split multiple JSON objects (since you used append mode)
# #     blocks = content.split("\n\n")

# #     for block in blocks:
# #         if block.strip():
# #             results_list.append(json.loads(block))

# # # 👉 Get latest run (last entry)
# # prev_results = results_list[-1]

# # # 🖨️ Pretty print
# # print("\n===== LOADED RESULTS =====")

# # print("\nExisting Features:")
# # print(results["existing_features"])

# # print("\nROC-AUC:")
# # print(results["roc_auc"])

# # print("\nTop Features:")
# # for k, v in results["top_features"].items():
# #     print(f"{k}: {v}")

# # print("\nClassification Report:")
# # print(results["classification_report"])


# import json

# # 📂 Load results file (handles multiple JSON blocks)
# results_list = []

# with open("results_2.json", "r") as f:
#     content = f.read().strip()
#     # Split multiple JSON objects (since you used append mode)
#     blocks = content.split("\n\n")

#     for block in blocks:
#         if block.strip():
#             results_list.append(json.loads(block))

# # 👉 Get latest run (last entry)
# prev_results = results_list[-1]

# # 🖨️ Pretty print
# print("\n===== LOADED RESULTS =====")

# print("\nExisting Features:")
# print(results["existing_features"])

# print("\nROC-AUC:")
# print(results["roc_auc"])

# print("\nTop Features:")
# for k, v in results["top_features"].items():
#     print(f"{k}: {v}")

# print("\nClassification Report:")
# print(results["classification_report"])

In [51]:
# user_df = get_userdf(prev_results)


In [52]:
# # what to load results for file here
# print(user_df.shape)
# results=run(user_df)
# with open("results_2.json", "a") as f:
#     f.write(json.dumps(results, indent=4))
#     f.write("\n\n")
# print("\n===== LLM INPUT =====")

# print("\nExisting Features:")
# print(llm_input["existing_features"])

# print("\nROC-AUC:")
# print(llm_input["roc_auc"])

# print("\nTop Features:")
# for k, v in llm_input["top_features"].items():
#     print(f"{k}: {v}")

# print("\nClassification Report:")
# print(llm_input["classification_report"])


In [53]:
# run(llm_input)


In [57]:
import json

log_file = "iteration_logs.json"
llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}
from getpass import getpass

getpass("Press ENTER to continue...")

for i in range(5):
    print(f"\n===== ITERATION {i} =====")

    user_df = get_userdf(llm_input)
    print(user_df.shape)
    #i need to press enter here to continue

    getpass("Press ENTER to continue...")
    llm_input = run(user_df)

    # Create log entry
    log_entry = {
        "iteration": i,
        "features": llm_input["existing_features"],
        "roc_auc": llm_input["roc_auc"],
        "classification_report": llm_input["classification_report"],
        "top_features": llm_input["top_features"]
    }

    # Append to file
    with open(log_file, "a") as f:
        f.write(json.dumps(log_entry, indent=4))
        f.write("\n\n")

    print("ROC-AUC:", llm_input["roc_auc"])


===== ITERATION 0 =====
Applying: user_max_to_avg_amount_ratio
Applying: user_night_transaction_ratio
Applying: user_unique_card_count
Applying: user_amount_coefficient_of_variation
Applying: user_zip_to_city_ratio
Applying: user_large_transaction_frequency
Applying: user_weekend_transaction_share
Applying: user_mcc_to_transaction_ratio
Applying: user_swipe_transaction_ratio
Applying: user_online_transaction_ratio
Applying: user_hourly_diversity_ratio
Applying: user_state_to_transaction_ratio
Applying: user_error_per_transaction_rate
Applying: user_amount_interquartile_range
Applying: user_avg_daily_velocity
Applying: user_active_year_span
Applying: user_mean_minus_median_amount
Applying: user_peak_hour_ratio
Applying: user_card_usage_entropy
Applying: user_holiday_season_ratio
(2000, 21)

=== Model Evaluation ===

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.87      0.73      0.79       131
           1       0.88      0.95